In [1]:
!pip install -U transformers datasets accelerate scikit-learn huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/NLP_Final_Project/Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.

In [2]:
from huggingface_hub import notebook_login
notebook_login() # Input HF token here

In [3]:
from datasets import load_dataset, concatenate_datasets

# 1. Load the Source Languages
print("Loading Hausa and Amharic datasets...")
dataset_hau = load_dataset("afrihate/afrihate", "hau")
dataset_amh = load_dataset("afrihate/afrihate", "amh")

# 2. Concatenate the sets
train_combined = concatenate_datasets([dataset_hau['train'], dataset_amh['train']])
test_source = concatenate_datasets([dataset_hau['test'], dataset_amh['test']])

# 3. Create Label Mappings (String -> Integer)
unique_labels = sorted(list(set(train_combined['label'])))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
print(f"Label mapping created: {label2id}")

def encode_labels(example):
    example['label'] = label2id[example['label']]
    return example

# Apply the mapping to both the combined train set and the test set
train_combined = train_combined.map(encode_labels)
test_source = test_source.map(encode_labels)

# 4. Create strict Train, Dev(Eval), and Test splits
# Splitting the combined train set: 85% Train, 15% Dev/Eval
train_eval_split = train_combined.train_test_split(test_size=0.15, seed=42)

train_data = train_eval_split['train']
eval_data = train_eval_split['test']
test_data = test_source # Held-out test set for the source languages

print(f"Train size: {len(train_data)} | Eval size: {len(eval_data)} | Test size: {len(test_data)}")

Loading Hausa and Amharic datasets...


README.md: 0.00B [00:00, ?B/s]

train.tsv: 0.00B [00:00, ?B/s]

dev.tsv: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Label mapping created: {'Abuse': 0, 'Hate': 1, 'Normal': 2}


Map:   0%|          | 0/8033 [00:00<?, ? examples/s]

Map:   0%|          | 0/1796 [00:00<?, ? examples/s]

Train size: 6828 | Eval size: 1205 | Test size: 1796


In [4]:
from transformers import AutoTokenizer

model_name = "Davlan/afro-xlmr-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Standardizing the text column. Adjust 'text' to 'tweet' if the dataset schema differs.
    text_column = "text" if "text" in examples else "tweet"
    return tokenizer(examples[text_column], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_eval = eval_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/6828 [00:00<?, ? examples/s]

Map:   0%|          | 0/1205 [00:00<?, ? examples/s]

Map:   0%|          | 0/1796 [00:00<?, ? examples/s]

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # zero_division=0 prevents warnings when the model hasn't learned all classes yet
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='macro',
        zero_division=0
    )

    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

If training on A100 GPU,

* Increase num_train_epochs to 3 (or more).

* Enable BFloat16: bf16 = True (leave fp16 = False).
A100 supports native bf16, which is more stable than fp16 and avoids gradient scaling issues.

* Optionally increase per_device_train_batch_size if GPU memory allows (e.g., 4 or 8) and reduce gradient_accumulation_steps accordingly to keep effective batch size similar.

* Remove max_steps (already removed) to let epochs control training.

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load model and inject label mappings
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    label2id=label2id,
    id2label=id2label
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",
    eval_steps=100,                # evaluate every 100 steps (adjust based on dataset size)
    save_strategy="steps",
    save_steps=100,
    logging_steps=50,
    save_total_limit=2,
    learning_rate=1e-5,
    warmup_steps=5,
    fp16=False,                    # T4: disable fp16 to avoid gradient unscaling issues
    bf16=False,                    # Set to True on A100 later
    max_grad_norm=1.0,             # Enable gradient clipping
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # effective batch size = 16
    per_device_eval_batch_size=1,
    num_train_epochs=1,            # Dry run: 1 epoch (~427 steps); increase later
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=42,                       # Reproducibility
    report_to="none"               # Disable wandb / other logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,0.000000,nan,0.314523,0.159512,0.104841,0.333333
200,0.000000,nan,0.314523,0.159512,0.104841,0.333333
300,0.000000,nan,0.314523,0.159512,0.104841,0.333333
400,0.000000,nan,0.314523,0.159512,0.104841,0.333333
427,0.000000,nan,0.314523,0.159512,0.104841,0.333333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=427, training_loss=5500.213114754099, metrics={'train_runtime': 864.5721, 'train_samples_per_second': 7.898, 'train_steps_per_second': 0.494, 'total_flos': 1590812212202496.0, 'train_loss': 5500.213114754099, 'epoch': 1.0})

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Evaluating on held-out test set...")

# Use predict() instead of evaluate() to avoid callback state issues
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Compute metrics manually
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, preds, average='macro', zero_division=0
)
acc = accuracy_score(labels, preds)

test_results = {
    "test_accuracy": acc,
    "test_f1": f1,
    "test_precision": precision,
    "test_recall": recall
}
print(test_results)

# Save final model to Google Drive
FINAL_MODEL_PATH = '/content/drive/MyDrive/NLP_Final_Project/Final_Source_Model'
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Final model saved to {FINAL_MODEL_PATH}")

Evaluating on held-out test set...


{'test_accuracy': 0.30902004454342985, 'test_f1': 0.15737983836665248, 'test_precision': 0.10300668151447662, 'test_recall': 0.3333333333333333}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to /content/drive/MyDrive/NLP_Final_Project/Final_Source_Model
